In [1]:
from setup import *

## Unit 4.1: Foundations of Predictive Modeling

### DAIR-3 Workshop, Summer 2026
<b>Presented by Suraj Rampure (rampure@umich.edu)</b>

---

### Introduction 👋

- My name is Suraj Rampure.<br>
<small>My first name is pronounced "sooh-rudge".</small>
- I am Teaching Faculty in Computer Science and Engineering at the University of Michigan.
- Recently, I've taught _Mathematics for Machine Learning_ and _Practical Data Science_.

### Infrastructure 💻

- All of the content I will present can be found in <br><center>https://github.com/DAIR3/DAIR3-Workshop/tree/main/resources/unit_4</center>
    
- Each "block" (4 total) has one Jupyter Notebook, which contains both the slides I will present and activities for you to work on.

- **Have these Jupyter Notebooks open in VSCode while I'm presenting!**<br><small>If you have trouble accessing or running them, let me know.</small>

- At the end of each block, you will write (1) write code and (2) submit reflections to ALICE.<br><center>https://utsa-alice.org</center>

<center></center>

### Before: Inference

- In Unit 3: **Rigorous Statistical Design**, you were developing scientific research aims and analytic plans.
- The emphasis was on **statistical inference**.<br><small>You studied estimation, effect sizes, uncertainty, and causal/mechanistic reasoning.</small>
- You were explicitly **not** trying to make "high prediction accuracy" the goal.

### Now: Prediction

- In Unit 4: **Designing Interpretable Predictive Models**, prediction **is** the goal.<br><small>The question shifts from "what can we infer about the relationships between variables?" to "what can we predict for new units, using only information available at prediction time?"</small>
- In our setting, models need to be interpretable and are subject to scrutiny by governing bodies.<br><small>We don't _just_ care about maximizing performance numbers.</small>
- We will start by introducing the foundations of predictive modeling, then focus on
    - Ensuring our models generalize well to unseen data.
    - Selecting covariates in an interpretable and reproducible manner.
    - Documenting our decisions.

### Taxonomy of machine learning

- Machine learning methods are often organized by the kind of feedback they learn from.
- Unit 4 lives mostly in **supervised learning**: we have examples with inputs and a target.
- Our main thread is **regression for prediction**, with interpretability and reproducibility constraints.

<center><img src="images/taxonomy.svg" alt="Taxonomy of machine learning methods, with Unit 4 focused on supervised regression for prediction." width=900></center>


## Regression in `sklearn`

---

### Loading the data

- Run the cell below to load in our toy dataset.
- Our predictors will be `departure_hour` and `day_of_month`; our target variable is commute time in `minutes`.
- Another term for predictor is **feature**.

In [97]:
df = pd.read_csv('data/commute-times.csv')
df['day_of_month'] = pd.to_datetime(df['date']).dt.day
df[['departure_hour', 'day_of_month', 'minutes']]

,departure_hour,day_of_month,minutes
0,10.816667,15,68.0
1,7.750000,16,94.0
2,8.450000,22,63.0
3,7.133333,23,100.0
4,9.150000,30,69.0
...,...,...,...
60,7.516667,27,85.0
61,7.550000,29,91.0
62,7.583333,4,68.0
63,7.450000,5,90.0


### Identifying the prediction problem

- **Goal**: Predict commute time in `minutes`.
- **Why?** To make commute time predictions for future days.
- **Unit of analysis**: one day.
- **How the data was collected**: logs from one individual across several years.

### Visualizing commute time

- Before fitting a model, we should visualize relationships between the predictor and target.

- Here, each point is one commute, with departure hour on the $x$-axis and commute time on the $y$-axis.


In [98]:
px.scatter(
    df,
    x='departure_hour',
    y='minutes',
    title='Commute time vs. departure hour',
    labels={
        'departure_hour': 'Departure hour',
        'minutes': 'Commute time (minutes)',
    },
    hover_data=['date', 'day'],
)

### `sklearn`

- `sklearn` (scikit-learn) is an industry-standard Python library for non-deep machine learning.

<center><img src='images/sklearn.png' width=20%></center>

- It interfaces with `numpy` arrays, and to an extent, `pandas` DataFrames.

- Huge benefit: the [documentation online](https://scikit-learn.org/stable/modules/classes.html) is excellent.


### The `LinearRegression` class

- `sklearn` comes with several subpackages, including `linear_model` and `tree`, each of which contains several classes of models.

- We'll start with the `LinearRegression` class from `linear_model`.

- **Important**: From [the documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression), we have:
> LinearRegression fits a linear model with coefficients w = (w1, …, wp) to minimize the residual sum of squares between the observed targets in the dataset, and the targets predicted by the linear approximation.

- It is important to understand the (hidden!) assumptions being made when fitting a model.

In [4]:
from sklearn.linear_model import LinearRegression

### Fitting a multiple linear regression model

- Let's aim to use `sklearn` to find the optimal **parameters** for the following model:

$$\begin{align*}\text{pred. commute}_i &= h(\text{departure hour}_i, \text{day of month}_i) \\ &= w_0 + w_1 \cdot \text{departure hour}_i + w_2 \cdot \text{day of month}_i \end{align*}$$

- First, we must instantiate a `LinearRegression` object and **train it**. By calling `fit`, we are saying "minimize mean squared error on this dataset and find $w_0^*$, $w_1^*$, and $w_2^*$.

- After fitting, we can access the best intercept ($w_0^*$) and coefficients ($w_1^*, w_2^*$).

In [99]:
model_multiple = LinearRegression()
model_multiple

LinearRegression()

In [101]:
# This line trains our model.
model_multiple.fit(X=df[['departure_hour', 'day_of_month']], y=df['minutes'])

LinearRegression()

### Accessing fit coefficients

- These coefficients tell us that the "best way" (according to squared loss) to make commute time predictions using a linear model is using:

$$\text{pred. commute}_i = 141.86 - 8.22 \cdot \text{departure hour}_i + 0.06 \cdot \text{day of month}_i$$

- This is the **plane of best fit** given historical data; it is not a causal statement.

- Does this say that departure time is a more useful predictor than day of the month? Not necessarily: it depends on the scale of the data.<br><small>What if I measured departure time in minutes from 12:00 instead of hours?</small>

In [103]:
model_multiple.intercept_, model_multiple.coef_ 

(141.86402699471932, array([-8.2233821 ,  0.05615985]))

<div class="alert alert-success">

### Discuss: Where did these optimal parameters come from?

</div>
    
- How did `sklearn` find these values? How would you find them on your own, without a pre-build software implementation?

In [127]:
model_multiple.intercept_, model_multiple.coef_ 

(141.86402699471932, array([-8.2233821 ,  0.05615985]))

### Making predictions

- Fit `LinearRegression` objects have a `predict` method, which can be used to predict commute times given a `departure_hour` and `day_of_month`.
- In modern ML/AI, this step is (confusingly) called "inference".

In [104]:
# What if I leave at 9:15AM on the 26th of the month?
model_multiple.predict(pd.DataFrame({
    'departure_hour': [9.25],
    'day_of_month': [26],
}))

array([67.25789852])

In [105]:
model_multiple.predict(pd.DataFrame({'departure_hour': [9.25], 'day_of_month': [26]}))

array([67.25789852])

In [106]:
departure_grid = np.linspace(df['departure_hour'].min(), df['departure_hour'].max(), 25)
day_grid = np.linspace(df['day_of_month'].min(), df['day_of_month'].max(), 25)
departure_mesh, day_mesh = np.meshgrid(departure_grid, day_grid)

prediction_grid = pd.DataFrame({
    'departure_hour': departure_mesh.ravel(),
    'day_of_month': day_mesh.ravel(),
})
minute_mesh = model_multiple.predict(prediction_grid).reshape(departure_mesh.shape)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=df['departure_hour'],
    y=df['day_of_month'],
    z=df['minutes'],
    mode='markers',
    name='Observed commutes',
    marker={'size': 5, 'opacity': 0.8},
))
fig.add_trace(go.Surface(
    x=departure_mesh,
    y=day_mesh,
    z=minute_mesh,
    name='Plane of best fit',
    colorscale='Oranges',
    opacity=0.55,
    showscale=False,
))
fig.update_layout(
    title='Plane of best fit',
    scene={
        'xaxis_title': 'Departure hour',
        'yaxis_title': 'Day of month',
        'zaxis_title': 'Commute time (minutes)',
    },
    margin={'l': 0, 'r': 0, 'b': 0, 't': 50},
)
fig

### `LinearRegression` class summary

| Property | Example | Description |
| --- | --- | --- |
| Initialize model parameters | `lr = LinearRegression()` | Create an empty linear regression model. |
| Fit the model to the data | `lr.fit(X, y)` | Determine regression coefficients from training data. |
| Use model for prediction | `lr.predict(X_new)` | Use the fitted regression equation to make predictions. |
| Access fitted attributes | `lr.coef_`, `lr.intercept_` | Inspect the fitted coefficients and intercept. |

### Evaluating model performance

- Since we're going to start to fit lots of different models to the commute times dataset, it'll be useful to compare their **mean squared errors**.

    $$\text{mean squared error} = \frac{1}{n} \sum_{i = 1}^n \left( y_i - h(x_i) \right)^2$$

    where $y_i$ is the $i$-th target value and $h(x_i)$ is the $i$-th predicted target value

- `sklearn` has a built-in `mean_squared_error` function.<br><small>Remember, the units of MSE are the units of $y$, squared. So the value below is 96.78 $\text{minutes}^2$.</small>

In [107]:
((df['minutes'] - model_multiple.predict(df[['departure_hour', 'day_of_month']])) ** 2).mean()

96.78730488437492

In [108]:
# Same result.
from sklearn.metrics import mean_squared_error
mean_squared_error(df['minutes'], model_multiple.predict(df[['departure_hour', 'day_of_month']])) 

96.78730488437492

### Comparing models

- Let's construct a dictionary to keep track of the MSEs we've seen so far.

- To compare, let's also fit and measure a simple linear model (only using `departure_hour`) and constant model (same prediction for all observations).

In [109]:
mse_dict = {}
mse_dict['departure_hour + day_of_month'] = mean_squared_error(
    df['minutes'], model_multiple.predict(df[['departure_hour', 'day_of_month']])
)

In [110]:
# Simple linear model.
model_simple = LinearRegression()
model_simple.fit(X=df[['departure_hour']], y=df['minutes'])
mse_dict['departure_hour'] = mean_squared_error(df['minutes'], model_simple.predict(df[['departure_hour']]))

In [111]:
# Constant model.
model_constant = df['minutes'].mean() # Best constant when using mean squared error is the mean.
mse_dict['constant'] = mean_squared_error(df['minutes'], np.ones(df.shape[0]) * model_constant)

<div class="alert alert-success">

### Discuss: Is adding more features always a good thing?

</div>


- As we can see, adding `day_of_month` as a feature **barely** reduced our model's MSE.

In [112]:
pd.Series(mse_dict).plot(kind='barh').update_layout(xaxis_title='Mean Squared Error', yaxis_title='Model')

- When might it be a bad idea to add more features?

## Generalization

---

### What's the point?

- The goal of building a predictive model is to make reliable predictions **on unseen data**.
- In order to ensure our models **generalize**, we can't just look at its performance on the data we used to fit it.

### Train-test splits 🚆

- Suppose we're choosing between many different models.

- We won't know whether a model has **overfit** to our sample unless we get to see how well it performs on a new sample from the same population.

- 💡**Idea**: **Split** our dataset into a <span style='color: blue'><b>training set</b></span> and <span style='color: orange'><b>test set</b></span>.

    <center><img src='images/train-test-first.png' width=700></center>

    How do we split the data? This is something we'll discuss shortly.

- For each model we're considering:
    - Use **only** the training set to fit that model (i.e. find $\vec{w}^*$).
    - Use the test set to evaluate that model's error (e.g. compute its MSE).

- Pick the model with the **best** test set performance.

- Why? The test set is like a new sample of data from the same population as the training data.

### Example: Polynomial regression

- To elaborate on this point, we'll use a small synthetic dataset.

- The data were generated from a nonlinear relationship with noise, so a polynomial regression model is a natural example.

In [113]:
sample_1 = make_polynomial_sample(n=100)
fig = px.scatter(x=sample_1['x'], y=sample_1['y'])
fig

### Polynomial fits on sample

- Before introducing a train/test split, let's treat the entire sample as training data. 
- Drag the slider to see how the fitted polynomial changes as the degree ranges from 1 to 30.

- A more flexible model can fit the training data more closely, but that does not automatically mean it will predict new data well.

In [114]:
degree_grid = np.arange(1, 31)
xs = pd.DataFrame({'x': np.linspace(sample_1['x'].min(), sample_1['x'].max(), 300)})

fig = px.scatter(sample_1, x='x', y='y', title='Polynomial fit: use the slider to change model flexibility')

for degree in degree_grid:
    model = make_pipeline(StandardScaler(), PolynomialFeatures(degree, include_bias=False), LinearRegression())
    model.fit(sample_1[['x']], sample_1['y'])
    fitted = model.predict(xs)
    fig.add_scatter(
        x=xs['x'],
        y=fitted,
        mode='lines',
        name=f'Degree {degree}',
        line={'color': 'orange', 'width': 4},
        visible=(degree == 1),
    )

steps = []
for i, degree in enumerate(degree_grid):
    visible = [True] + [False] * len(degree_grid)
    visible[i + 1] = True
    steps.append({
        'method': 'update',
        'label': str(degree),
        'args': [
            {'visible': visible},
            {'title': f'Polynomial fit: degree {degree}'},
        ],
    })

fig.update_layout(
    showlegend=False,
    sliders=[{
        'active': 0,
        'currentvalue': {'prefix': 'Degree: '},
        'pad': {'t': 40},
        'steps': steps,
    }],
)
fig

### Implementing a train-test split

In [115]:
from sklearn.model_selection import train_test_split

In [121]:
# train_test_split?

In [122]:
X = sample_1[['x']] 
y = sample_1['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23) 

In [124]:
print('Rows in X_train:', X_train.shape[0])
display(X_train.head())
print('Rows in X_test:', X_test.shape[0])
display(X_test.head())

Rows in X_train: 80


,x
85,3.585859
28,-2.171717
8,-4.191919
11,-3.888889
63,1.363636


Rows in X_test: 20


,x
26,-2.373737
80,3.080808
82,3.282828
68,1.868687
77,2.777778


### Training on _only_ the training data

- Ignore most of the code below; it involves syntax that we'll see later on.

In [125]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
train_errs = []
test_errs = []
for d in range(1, 31):
    pl = make_pipeline(StandardScaler(), PolynomialFeatures(d, include_bias=False), LinearRegression())
    pl.fit(X_train, y_train)
    train_errs.append(mean_squared_error(y_train, pl.predict(X_train)))
    test_errs.append(mean_squared_error(y_test, pl.predict(X_test)))
errs = pd.DataFrame({'Train Error': train_errs, 'Test Error': test_errs})
errs

,Train Error,Test Error
0,20.240161,25.063581
1,13.202878,15.522270
2,10.987288,10.323537
3,10.982120,10.418200
4,10.926117,10.229075
5,10.784634,10.610924
6,10.777538,10.641458
7,10.702730,10.628845
8,10.444038,11.127926
9,9.425691,10.104174


<div class="alert alert-success">

### Discuss: Which degree should we choose?

</div>

In [128]:
errs.index = errs.index + 1
fig = px.line(errs.iloc[:-1])
fig.update_layout(showlegend=True, xaxis_title='Polynomial Degree', yaxis_title='Mean Squared Error')
fig

### Training error vs. test error

- The pattern we saw in the previous example is true more generally.

<center><img src='images/tt-errors.png' width=600></center>

- We pick the models at the "valley" of test error.

- Note that training error **tends** to underestimate test error, but it doesn't have to – i.e., it is possible for test error to be lower than training error (say, if the test set is "easier" to predict than the training set).

- The results – and the bias-variance tradeoff more generally – hold true for "classic" machine learning models, like the ones we're studying here. But in deep neural networks, this pattern is often violated; extremely complex models can have low test error as well.<br><small>This phenomenon is known as "double descent"; learn more [**here**](https://en.wikipedia.org/wiki/Double_descent).</small>

### How is this relevant to biomedical examples?

- It _may_ be the case that you need to tune a **hyperparameter**, like polynomial degree. The theory we just discussed applies there.
- Instead, it may be the case that you have several candidate sets of features that you'd like to choose between.

### Conducting train-test splits

- Recall, <span style='color: blue'><b>training data</b></span> is used to fit our model, and <span style='color: orange'><b>test data</b></span> is used to evaluate our model.

<center><img src='images/train-test-first.png' width=40%></center>

- **Random split**: randomly assign rows to train/test. This is the default `train_test_split` behavior, and it is a good first syntax pattern.
- **Time split**: train on earlier observations and test on later observations. Use this when the goal is **forecasting future data**.
- **Group split**: keep entire groups together, e.g. train on some counties and test on held-out counties. Use this when rows within a group are correlated.
- **Stratified split**: preserve the distribution of an important categorical variable or rare outcome across train/test sets.
- The split should match the way the model will actually be used.

### Data leakage

- **Data leakage** occurs when information that would not be available at prediction time is used while fitting, tuning, or evaluating a model.
- Leakage can make held-out performance look better than it really is, even if we used a train/test split.
- Common biomedical examples:
    - preprocessing the full dataset before splitting, so test-set summaries influence the training data;
    - letting records from the same patient, hospital, county, or time period appear in both train and test;
    - using variables recorded after the outcome, or variables that are downstream consequences of the outcome;
    - selecting features or model settings after repeatedly checking test performance.
- Rule of thumb: every choice that learns from data should be made using training data only, then applied unchanged to the test set.

### But wait...

- With our current strategy, we are choosing the model that creates the model that **performs best on the test set**.

- As such, we are **overfitting to the test set** – the best model on the test set might not be the best model on a totally unseen dataset.

- It seems like we need **another** split.

- The real solution is called **cross-validation**. We'll discuss it later in Unit 4.

## Now: the real birthweight dataset

---


### Reintroducing the birthweight data

- Now that we have seen the `sklearn` syntax and train/test split ideas on toy examples, we can return to the real dataset from earlier sessions.
- Unit 3 used quantitative `birthweight`, measured in grams, as the outcome.
- We will use the same target variable here, but now the goal is predictive performance on held-out births.
- The next cell uses the same prepared `1971.csv.gz` file expected by the Unit 3 notebook.

In [138]:
try:
    births = load_birthweight_1971()
    print(f"Loaded {births.shape[0]:,} births and {births.shape[1]:,} columns.")
    display(births.sample(5))
except FileNotFoundError as err:
    births = None
    print(err)

Loaded 1,781,774 births and 14 columns.


,year,state,county,smsa,sex,dadrace,momrace,momage,birthorder,dadage,birthweight,plurality,interval,popsize
117771,1971,5,019,114,female,Black,Black,21,2.0,23.0,3969.0,1,30.0,9
57471,1971,4,026,000,male,White,White,20,1.0,20.0,2410.0,1,NaN,9
443967,1971,14,016,042,female,White,White,27,3.0,27.0,2750.0,1,67.0,9
482010,1971,14,058,054,female,Black,Black,20,1.0,23.0,3232.0,1,NaN,9
1226349,1971,36,060,000,male,White,White,25,2.0,30.0,2637.0,1,55.0,9


### Identifying the prediction problem

- **Goal**: Predict `birthweight` in grams.
- **Why?** To make commute time predictions for future days.
- **Unit of analysis**: one baby's birth.
- **How the data was collected**: NCHS collects data on all births in the US in each year.

<div class="alert alert-success">

### Activity: Fit a baseline model

</div>

Using the `births` DataFrame below,
1. Perform a train test split of size 20%, where the features used are `momage`, `dadage`, `plurality`, and `birthorder`.
1. Find the training and test set MSE of the model's performance.
1. On ALICE, include the following:
    1. A screenshot of your code, and both MSEs.
    1. Your thoughts on whether a random train/test split makes sense for the current prediction task, and thoughts on how data leakage may be an issue present.

In [144]:
births

,year,state,county,smsa,sex,dadrace,momrace,momage,birthorder,dadage,birthweight,plurality,interval,popsize
0,1971,1,001,000,male,White,White,20,1.0,22.0,4196.0,1,NaN,9
1,1971,1,002,128,male,White,White,27,3.0,27.0,2920.0,1,NaN,9
2,1971,1,002,128,female,Black,Black,20,3.0,30.0,2892.0,1,NaN,9
3,1971,1,002,128,male,White,White,28,5.0,32.0,3232.0,1,NaN,9
4,1971,1,002,128,male,White,White,30,3.0,26.0,3175.0,1,NaN,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1781769,1971,51,016,000,female,White,White,20,2.0,24.0,3175.0,1,NaN,9
1781770,1971,51,012,000,male,White,White,28,3.0,28.0,3430.0,1,53.0,9
1781771,1971,51,013,000,female,White,White,19,1.0,21.0,3600.0,1,NaN,9
1781772,1971,51,013,000,female,White,White,18,1.0,21.0,3260.0,1,NaN,9


In [142]:
# Your code goes here...